# VKR PatchTST + ICEEMDAN: прогон бэктеста на Colab GPU

**Перед запуском:**
1. Runtime → Change runtime type → **L4 GPU** (или A100).
2. Загрузите `vkr_patch_colab.zip` одним из способов:
   - в сессию: панель слева → Files → Upload (быстро, но файл живёт до конца сессии);
   - на Google Drive в корень `MyDrive` (надёжнее).
3. Выполняйте ячейки сверху вниз.

**Важно:** держите вкладку открытой (без Colab Pro фоновое выполнение не гарантируется).
Результаты каждой стратегии сохраняются сразу после её завершения, но сессионный диск
очищается при отключении — поэтому в конце есть ячейка копирования результатов на Drive.

In [ ]:
# GPU на месте?
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'GPU не включён: Runtime -> Change runtime type -> L4'
print('torch', torch.__version__, '| cuda:', torch.cuda.get_device_name(0))

In [ ]:
# Google Drive (рекомендуется: туда уйдут результаты)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Распаковка проекта
import os, zipfile

candidates = ['/content/vkr_patch_colab.zip', '/content/drive/MyDrive/vkr_patch_colab.zip']
zip_path = next((p for p in candidates if os.path.exists(p)), None)
assert zip_path, 'Не найден vkr_patch_colab.zip: загрузите его в /content или в корень Drive'
print('Архив:', zip_path)

os.makedirs('/content/VKR_Patch', exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/VKR_Patch')
%cd /content/VKR_Patch
!ls && echo '---' && ls data/raw && echo 'кэш декомпозиций:' && ls data/cache/iceemdan 2>/dev/null | wc -l

In [ ]:
# Зависимости (torch/pandas/sklearn/matplotlib уже стоят в Colab)
!pip install -q EMD-signal statsforecast yfinance

In [ ]:
# Проверка пайплайна + бенчмарк одного окна (даст оценку времени всего прогона)
import sys, time, yaml
sys.path.insert(0, 'src')
import pandas as pd

cfg = yaml.safe_load(open('config/config.yaml'))
print('mode:', cfg['models']['patchtst']['mode'], '| data_end:', cfg['backtest']['data_end'],
      '| n_workers:', cfg['models']['patchtst']['n_workers'], '(на CUDA оставить 1)')

returns = pd.read_csv('data/raw/log_returns.csv', index_col=0, parse_dates=True)
data_end = cfg['backtest'].get('data_end')
if data_end:
    returns = returns.loc[:data_end]
print('данные:', returns.shape, returns.index[0].date(), '—', returns.index[-1].date())

import backtesting.backtest_patchtst as bt
t0 = time.time(); bt.forecast_returns_patchtst(returns.iloc[:1260], horizon=21); dt = time.time() - t0
steps = (len(returns) - 1260 - 21) // 21 + 1
print(f'Окно full-режима: {dt:.0f} с (на M4 Air было ~48 с)')
print(f'Окон в прогоне: {steps}. Оценка: PatchTST ~{steps*dt/60:.0f} мин, ICEEMDAN ~x2-3 от этого')

## Декомпозиции ICEEMDAN (CPU)

Если в архиве уже есть кэш (`data/cache/iceemdan`, см. счётчик выше — для полного покрытия
нужно ~5000 файлов), этот шаг пройдёт мгновенно. Иначе недостающие окна досчитаются
параллельно на CPU Colab (~30–60 мин). Без этого шага бэктест посчитает их сам,
но последовательно — медленнее.

In [ ]:
!python scripts/precompute_decompositions.py --all

## Запуск бэктеста

`STRATEGIES`: пусто = все четыре; `'3,4'` = только PatchTST и PatchTST+ICEEMDAN;
`'4'` = только ICEEMDAN (для итераций тюнинга).

Период задаётся в `config/config.yaml` → `backtest.data_end`:
`"2015-02-01"` = тюнинг по validation, `null` = полный финальный прогон
(поменять можно ячейкой ниже). Прогресс пишется и в `results/run_log_*.txt`.

In [ ]:
# (опционально) поменять период, не трогая архив
DATA_END = 'keep'   # 'keep' = как в config | '2015-02-01' = validation | 'null' = полный период

if DATA_END != 'keep':
    import re
    text = open('config/config.yaml').read()
    value = 'null' if DATA_END == 'null' else f'"{DATA_END}"'
    text = re.sub(r'data_end: (null|"[0-9-]+")', f'data_end: {value}', text, count=1)
    open('config/config.yaml', 'w').write(text)
import yaml
print('data_end сейчас:', yaml.safe_load(open('config/config.yaml'))['backtest']['data_end'])

In [ ]:
STRATEGIES = ''   # '' = все; '3,4' = обе PatchTST; '4' = только ICEEMDAN
!printf "n\n{STRATEGIES}\n" | MPLBACKEND=Agg python run_all.py

In [ ]:
# Итоги: сводные таблицы последнего прогона
import glob, pandas as pd
latest = sorted(glob.glob('results/comparison_full_*.csv'))[-1]
ts = latest.split('comparison_full_')[1].replace('.csv', '')
print('Прогон:', ts)
for period in ['validation', 'holdout', 'full']:
    path = f'results/comparison_{period}_{ts}.csv'
    df = pd.read_csv(path, index_col=0)
    cols = ['Sharpe Ratio', 'Sharpe Ratio (net)', 'Calmar Ratio (net)', 'Max Drawdown (net)', 'Avg Turnover']
    print(f'\n=== {period} ===')
    display(df[cols].round(3))

In [ ]:
# Сохранение на Drive: результаты + кэш декомпозиций (пригодится для следующих прогонов)
import shutil, datetime, os
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
results_zip = shutil.make_archive(f'/content/results_{stamp}', 'zip', 'results')

drive_dir = '/content/drive/MyDrive/VKR_results'
if os.path.isdir('/content/drive/MyDrive'):
    os.makedirs(drive_dir, exist_ok=True)
    shutil.copy(results_zip, drive_dir)
    cache_zip = shutil.make_archive(f'/content/iceemdan_cache_{stamp}', 'zip', 'data/cache/iceemdan')
    shutil.copy(cache_zip, drive_dir)
    print('Сохранено на Drive:', drive_dir)
else:
    from google.colab import files
    files.download(results_zip)